# Package a Customer H2O Model

This intake notebook supports two prepopulated customer profiles: a native H2O binary and a MOJO ZIP. It detects the artifact format from the selected file, detects whether a complete golden-data pair is present, validates the profile contract, and stages one canonical bundle for notebooks `02` through `05`.

## Before you run it

1. Set `CUSTOMER_MODEL_PROFILE` in Cell 2 to either `"mojo"` or `"binary"`.
2. Open the matching folder under `data/h2o/customer_bundle/profiles/`.
3. Replace the sample model with exactly one customer model artifact.
4. Replace both `golden_input.csv` and `golden_expected.csv`, or remove both when the customer has no golden dataset.
5. Update `profile.json` with the runtime and schema values that cannot be inferred from the artifact.

MOJO producer version, MOJO version, feature order, and target are read from `model.ini`. Native binaries require those values in `profile.json` because they cannot be inspected without loading the producer runtime.

The selected source profile is copied to ignored `outputs/h2o_customer_bundle/`. Nothing is uploaded to Azure in this notebook.

**Source:** Adapted from this repository's H2O reference and onboarding notebooks.

In [1]:
CUSTOMER_MODEL_PROFILE = "binary"

In [2]:
from pathlib import Path
import os
import sys

import pandas as pd
from dotenv import load_dotenv

notebook_file = globals().get("__vsc_ipynb_file__")
search_start = (
    Path(notebook_file).resolve().parent
    if notebook_file
    else Path.cwd().resolve()
)
for candidate in (search_start, *search_start.parents):
    if (candidate / ".env.example").is_file() and (candidate / "outputs").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")

load_dotenv(WORKSHOP_ROOT / ".env", override=True)

def enabled(name: str) -> bool:
    return os.getenv(name, "false").lower() in {"1", "true", "yes"}

sys.path.insert(0, str(WORKSHOP_ROOT / "src/h2o"))
from customer_profiles import prepare_customer_bundle

print(f"Workshop root: {WORKSHOP_ROOT}")
print(f"Selected customer profile: {CUSTOMER_MODEL_PROFILE}")

Workshop root: /home/azureuser/MLOPs-AzureML-workshop-277d1b6/workshop
Selected customer profile: binary


## 1. Select and stage a customer profile

The selected profile folder must contain `profile.json` and exactly one model artifact. The notebook detects whether that artifact is a native binary or MOJO ZIP and detects golden data only when both canonical CSV files are present.

The selected files are copied into `outputs/h2o_customer_bundle/`, which is the canonical handoff consumed by notebooks `02` through `05`.

In [3]:
RUN_GOLDEN_VALIDATION = os.getenv(
    "H2O_CUSTOMER_RUN_GOLDEN_VALIDATION",
    "true",
).lower() in {"1", "true", "yes"}
REQUIRE_GOLDEN_VALIDATION = enabled("H2O_CUSTOMER_REQUIRE_GOLDEN_VALIDATION")

prepared = prepare_customer_bundle(
    WORKSHOP_ROOT,
    CUSTOMER_MODEL_PROFILE,
    run_golden_validation=RUN_GOLDEN_VALIDATION,
    require_golden_validation=REQUIRE_GOLDEN_VALIDATION,
)

PROFILE_DIR = Path(prepared["profile_dir"])
BUNDLE_DIR = Path(prepared["bundle_dir"])
MODEL_PATH = Path(prepared["model_path"])
manifest_path = Path(prepared["manifest_path"])
manifest = prepared["manifest"]
bundle_summary = prepared["summary"]

MODEL_NAME = manifest["model_name"]
ENVIRONMENT_NAME = manifest["environment_name"]
MODEL_VERSION = str(manifest["model_version"])
MODEL_FORMAT = manifest["model_format"]
MODEL_H2O_VERSION = manifest["h2o_version"]
MOJO_VERSION = manifest.get("mojo_version")
RUNTIME_H2O_VERSION = manifest["runtime_h2o_version"]
PYTHON_VERSION = manifest["python_version"]
JAVA_VERSION = manifest["java_version"]
H2O_PIP_SPEC = manifest["h2o_pip_spec"]
FEATURES = manifest["features"]
CATEGORICAL_FEATURES = manifest["categorical_features"]
TARGET = manifest["target"]
GOLDEN_PROVIDED = manifest["golden_data"]["provided"]
RUN_GOLDEN_VALIDATION = manifest["golden_data"]["validate"]
REQUIRE_GOLDEN_VALIDATION = manifest["golden_data"]["required"]
GOLDEN_INPUT_PATH = (
    BUNDLE_DIR / manifest["golden_data"]["input_file"]
    if GOLDEN_PROVIDED
    else None
)
GOLDEN_EXPECTED_PATH = (
    BUNDLE_DIR / manifest["golden_data"]["expected_file"]
    if GOLDEN_PROVIDED
    else None
)

display(
    pd.DataFrame(
        {
            "setting": [
                "selected profile",
                "source folder",
                "staged bundle",
                "model file",
                "Azure ML model name",
                "Azure ML environment name",
                "detected format",
                "producer H2O",
                "MOJO format",
                "runtime H2O",
                "Python",
                "Java",
                "target",
                "features",
                "categorical features",
                "golden data",
                "golden rows",
                "run golden validation",
                "golden validation required",
            ],
            "value": [
                prepared["profile"],
                str(PROFILE_DIR),
                str(BUNDLE_DIR),
                MODEL_PATH.name,
                MODEL_NAME,
                ENVIRONMENT_NAME,
                MODEL_FORMAT,
                MODEL_H2O_VERSION,
                MOJO_VERSION or "not applicable",
                RUNTIME_H2O_VERSION,
                PYTHON_VERSION,
                JAVA_VERSION,
                TARGET,
                ", ".join(FEATURES),
                ", ".join(CATEGORICAL_FEATURES) or "none",
                "provided" if GOLDEN_PROVIDED else "not provided",
                prepared["golden_rows"],
                RUN_GOLDEN_VALIDATION,
                REQUIRE_GOLDEN_VALIDATION,
            ],
        }
    )
)

,setting,value
0,selected profile,binary
1,source folder,/home/azureuser/MLOPs-AzureML-workshop-277d1b6...
2,staged bundle,/home/azureuser/MLOPs-AzureML-workshop-277d1b6...
3,model file,taxi-fare-gbm
4,Azure ML model name,workshop-h2o-customer-binary
5,Azure ML environment name,workshop-h2o-customer-binary-environment
6,detected format,h2o_binary
7,producer H2O,3.46.0.12
8,MOJO format,not applicable
9,runtime H2O,3.46.0.12


## 2. Inspect the detected golden data

Golden data is detected by filename inside the selected profile. Both `golden_input.csv` and `golden_expected.csv` must exist, or both must be absent.

When present, the input columns must match the detected or declared feature order, and the expected CSV must contain one `predict` column with the same non-zero row count.

In [4]:
if GOLDEN_PROVIDED:
    golden_input = pd.read_csv(GOLDEN_INPUT_PATH)
    golden_expected = pd.read_csv(GOLDEN_EXPECTED_PATH)
    print(f"Golden rows: {len(golden_input)}")
    print(f"Prediction type: {manifest['prediction_type']}")
    display(golden_input.head())
    display(golden_expected.head())
else:
    print("Golden data was not found; runtime parity will be skipped.")

Golden rows: 20
Prediction type: number


,vendorID,passengerCount,tripDistance,paymentType,pickupHour
0,1,2,11.80,2,14
1,1,1,5.70,1,0
2,2,2,3.18,2,20
3,2,1,1.96,1,21
4,1,1,1.10,2,21


,predict
0,34.823667
1,20.211054
2,13.282541
3,9.505314
4,7.076866


## 3. Review the generated manifest

The staged manifest records the detected model format, selected runtime, schema, optional golden-data state, and SHA-256 checksum for every staged artifact. This is the handoff contract used by the remaining customer notebooks.

In [5]:
print(f"Manifest written to: {manifest_path}")
display(manifest)

Manifest written to: /home/azureuser/MLOPs-AzureML-workshop-277d1b6/workshop/outputs/h2o_customer_bundle/model_manifest.json


{'profile': 'binary',
 'model_name': 'workshop-h2o-customer-binary',
 'environment_name': 'workshop-h2o-customer-binary-environment',
 'endpoint_name': 'h2o-customer-binary-endpoint',
 'deployment_name': 'blue',
 'input_data_name': 'workshop-h2o-customer-binary-input',
 'experiment_name': 'workshop-h2o-customer-binary-scoring',
 'model_version': '1',
 'model_format': 'h2o_binary',
 'h2o_version': '3.46.0.12',
 'runtime_h2o_version': '3.46.0.12',
 'h2o_pip_spec': 'h2o==3.46.0.12',
 'python_version': '3.12',
 'java_version': '17',
 'model_file': 'taxi-fare-gbm',
 'target': 'fareAmount',
 'features': ['vendorID',
  'passengerCount',
  'tripDistance',
  'paymentType',
  'pickupHour'],
 'categorical_features': ['vendorID', 'paymentType'],
 'prediction_type': 'number',
 'files': {'taxi-fare-gbm': '995303cdb3e1462cae499693cca47ffc3d23b63a43ea35c6897b6d566124f0a3',
  'golden_input.csv': 'a0f4f80234d7877b17c7169ac82b5ed34286d04d2caa877aa6e316c5060e15ef',
  'golden_expected.csv': '928ffb70ee430f

## 4. Confirm the staged bundle

The shared packager has already reloaded the manifest and validated the detected format, runtime contract, schema, optional golden pair, and every checksum.

In [6]:
display(bundle_summary)

{'bundle_dir': '/home/azureuser/MLOPs-AzureML-workshop-277d1b6/workshop/outputs/h2o_customer_bundle',
 'model_name': 'workshop-h2o-customer-binary',
 'model_version': '1',
 'model_format': 'h2o_binary',
 'h2o_version': '3.46.0.12',
 'runtime_h2o_version': '3.46.0.12',
 'mojo_version': None,
 'model_category': None,
 'model_file': 'taxi-fare-gbm',
 'features': ['vendorID',
  'passengerCount',
  'tripDistance',
  'paymentType',
  'pickupHour'],
 'golden_provided': True,
 'golden_validation_enabled': True,
 'golden_required': False,
 'golden_input_file': 'golden_input.csv',
 'golden_expected_file': 'golden_expected.csv',
 'golden_rows': 20,
 'checksums': 'passed',
 'packaging_validation': 'passed',
 'golden_validation': 'pending'}

## 5. Defer prediction parity to the target runtime

Packaging is complete. Notebook `03` creates the profile-specific Azure ML environment. When golden files were detected, notebook `04` deploys with zero traffic and compares predictions before optional promotion. When they were absent, runtime parity is skipped unless explicitly required.

In [7]:
print("Packaging validation passed.")
print(f"Target runtime: h2o=={RUNTIME_H2O_VERSION}, Python {PYTHON_VERSION}, Java {JAVA_VERSION}")
print(
    "Runtime golden parity remains pending until notebook 04."
    if GOLDEN_PROVIDED and RUN_GOLDEN_VALIDATION
    else "Runtime golden parity is skipped by configuration."
    if GOLDEN_PROVIDED
    else "Runtime golden parity is optional and no golden data was provided."
)

Packaging validation passed.
Target runtime: h2o==3.46.0.12, Python 3.12, Java 17
Runtime golden parity remains pending until notebook 04.


## Expected Result

The selected binary or MOJO profile is auto-detected, validated, and staged under `outputs/h2o_customer_bundle/`. Golden data is included only when both canonical CSV files exist.

Next: `02_register_model.ipynb`.